In [1]:
from pathlib import Path
import pandas as pd
import pickle
import numpy as np

In [2]:
with open("../Data/all_random_pc_splits.pkl", "rb") as f:
    pc_splits = pickle.load(f)

# Print all unique column names that contain 'spanin'
spanin_cols = [c for c in pc_splits.columns if 'spanin' in c.lower()]
print(spanin_cols)

['split_Campylobacter_spanin', 'split_Caulobacter_spanin', 'split_Cellulophaga_spanin', 'split_Citrobacter_spanin', 'split_Clostridioides_spanin', 'split_Clostridium_spanin', 'split_Corynebacterium_spanin', 'split_Cronobacter_spanin', 'split_Dickeya_spanin', 'split_Escherichia_spanin', 'split_Edwardsiella_spanin', 'split_Enterobacter_spanin', 'split_Enterococcus_spanin', 'split_Erwinia_spanin', 'split_Flavobacterium_spanin', 'split_Gordonia_spanin', 'split_Haloarcula_spanin', 'split_Halorubrum_spanin', 'split_Klebsiella_spanin', 'split_Lacticaseibacillus_spanin', 'split_Lactobacillus_spanin', 'split_Lactococcus_spanin', 'split_Listeria_spanin', 'split_Microbacterium_spanin', 'split_Microcystis_spanin', 'split_Mycobacterium_spanin', 'split_Paenibacillus_spanin', 'split_Pantoea_spanin', 'split_Pectobacterium_spanin', 'split_Pelagibacter_spanin', 'split_Prochlorococcus_spanin', 'split_Propionibacterium_spanin', 'split_Proteus_spanin', 'split_Providencia_spanin', 'split_Pseudoalteromonas_s

In [3]:
with open("../Data/all_random_pc_splits.pkl", "rb") as f:
    pc_splits = pickle.load(f)

print(pc_splits.shape)
pc_splits.head()

(1853074, 2160)


,split_Campylobacter_lysin,split_Campylobacter_endolysin,split_Campylobacter_val,split_Caulobacter_lysin,split_Caulobacter_endolysin,split_Caulobacter_val,split_Cellulophaga_lysin,split_Cellulophaga_endolysin,split_Cellulophaga_val,split_Citrobacter_lysin,...,split_Providencia_val,split_Serratia_lysin,split_Serratia_endolysin,split_Serratia_val,split_Pelagibacter_lysin,split_Pelagibacter_endolysin,split_Pelagibacter_val,split_Shewanella_lysin,split_Shewanella_endolysin,split_Shewanella_val
proteinID,,,,,,,,,,,,,,,,,,,,,
AB002632_00001,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AB002632_00002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AB002632_00003,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AB002632_00004,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AB002632_00005,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Inspecting one split folder

In [4]:
folder0 = Path("../Data/random_pc_r_5_da_2_bal_s_0")

files = sorted(folder0.glob("*.pkl"))

print(f"Number of prediction files: {len(files)}")

for f in files[:10]:
    print(f.name)

Number of prediction files: 81
all_preds_gramneg_DNA-associated.pkl
all_preds_gramneg_DNA-associated_1.pkl
all_preds_gramneg_DNA_polymerase.pkl
all_preds_gramneg_RNA-associated.pkl
all_preds_gramneg_adsorption-related.pkl
all_preds_gramneg_annealing.pkl
all_preds_gramneg_anti-restriction.pkl
all_preds_gramneg_baseplate.pkl
all_preds_gramneg_capsid.pkl
all_preds_gramneg_cell_wall_depolymerase.pkl


Real prediction columns

In [5]:
all_pred_columns = []

for f in files:
    df = pd.read_pickle(f)

    all_pred_columns.extend(df.columns.tolist())

all_pred_columns = list(dict.fromkeys(all_pred_columns))

print("Total prediction columns:", len(all_pred_columns))
print(all_pred_columns[:20])

Total prediction columns: 2036
['Xanthomonas_DNA-associated', 'Edwardsiella_DNA-associated', 'Pseudoalteromonas_DNA-associated', 'Roseobacter_DNA-associated', 'Halorubrum_DNA-associated', 'Synechococcus_DNA-associated', 'Sulfolobus_DNA-associated', 'Sinorhizobium_DNA-associated', 'Shigella_DNA-associated', 'Yersinia_DNA-associated', 'Flavobacterium_DNA-associated', 'Prochlorococcus_DNA-associated', 'Haloarcula_DNA-associated', 'Erwinia_DNA-associated', 'Proteus_DNA-associated', 'Citrobacter_DNA-associated', 'Rhizobium_DNA-associated', 'Stenotrophomonas_DNA-associated', 'Campylobacter_DNA-associated', 'Pantoea_DNA-associated']


#### Verifying matching splits

In [6]:
missing = []

for col in all_pred_columns:

    split_col = "split_" + col

    if split_col not in pc_splits.columns:
        missing.append(col)

print("Missing:", len(missing))

if len(missing):
    print(missing[:20])

Missing: 0


#### Estimating the matrix size

In [7]:
n_proteins = len(pc_splits)
n_models = len(all_pred_columns)

print(n_proteins)
print(n_models)

gb = n_proteins * n_models * 4 / 1024**3

print(f"float32 matrix size ≈ {gb:.2f} GB")

1853074
2036
float32 matrix size ≈ 14.05 GB


In [8]:
sample_ids = pc_splits.index[:20]

for pid in sample_ids:
    print(pid)

AB002632_00001
AB002632_00002
AB002632_00003
AB002632_00004
AB002632_00005
AB002632_00006
AB002632_00007
AB002632_00008
AB002632_00009
AB002632_00010
AB008550_00001
AB008550_00002
AB008550_00003
AB008550_00004
AB008550_00005
AB008550_00006
AB008550_00007
AB008550_00008
AB008550_00009
AB008550_00010


#### Total proteins and phages

In [9]:
accessions = pd.Index(
    [pid.rsplit("_", 1)[0] for pid in pc_splits.index]
)

print("Proteins:", len(accessions))
print("Unique phages:", accessions.nunique())

Proteins: 1853074
Unique phages: 18474


In [10]:
meta_df = pd.read_csv(
    "../Data/metadata.csv",
    index_col="proteinID"
)

In [11]:
sample = pd.read_pickle(
    "../Data/random_pc_r_5_da_2_bal_s_0/all_preds_gramneg_capsid.pkl"
)

print(sample.shape)

sample.head()

(44804, 38)


,Vibrio_capsid,Pseudomonas_capsid,Salmonella_capsid,Microcystis_capsid,Escherichia_capsid,Ralstonia_capsid,Klebsiella_capsid,Xanthomonas_capsid,Edwardsiella_capsid,Pseudoalteromonas_capsid,...,Caulobacter_capsid,Dickeya_capsid,Cronobacter_capsid,Cellulophaga_capsid,Rhodobacter_capsid,Pectobacterium_capsid,Providencia_capsid,Serratia_capsid,Pelagibacter_capsid,Shewanella_capsid
proteinID,,,,,,,,,,,,,,,,,,,,,
AB002632_00003,0.892029,0.006306,0.009383,0.081754,0.041862,0.003088,0.015248,0.004815,0.000388,0.006762,...,0.007589,0.025637,0.003450,0.002163,0.000379,0.000957,0.024976,0.001247,0.000841,0.006299
AB002632_00005,0.884198,0.039370,0.016259,0.015769,0.091740,0.374117,0.007287,0.480330,0.003258,0.004493,...,0.002441,0.002847,0.002345,0.004689,0.000276,0.004016,0.009124,0.000302,0.001409,0.137713
AB002632_00006,0.960939,0.002251,0.005723,0.007318,0.034375,0.000299,0.008497,0.001767,0.003222,0.001731,...,0.000707,0.002753,0.001835,0.005969,0.000040,0.004501,0.005529,0.001472,0.002281,0.019812
AB002632_00007,0.887575,0.009491,0.011740,0.021452,0.247610,0.040026,0.000952,0.375594,0.001234,0.001174,...,0.005451,0.006769,0.002275,0.001705,0.000650,0.002436,0.008512,0.000485,0.001534,0.379902
AB008550_00005,0.003117,0.052489,0.009547,0.012469,0.173586,0.781633,0.570038,0.000270,0.000731,0.447287,...,0.007142,0.000516,0.022271,0.000672,0.002136,0.008286,0.017155,0.003673,0.013043,0.015565


In [12]:
print(sample.index.equals(pc_splits.index))

False


In [13]:
sample = pd.read_pickle(
    "../Data/random_pc_r_5_da_2_bal_s_0/all_preds_gramneg_capsid.pkl"
)

print(sample.index.nunique())
print(len(sample.index))

44804
44804


In [14]:
DATA_DIR = Path("../Data")
SPLITS_PATH = DATA_DIR / "all_random_pc_splits.pkl"
METADATA_PATH = DATA_DIR / "metadata.csv"
OUTPUT_DIR = DATA_DIR / "processed_chunks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [15]:


# # Define your safe row chunk size (adjust lower if 100k still pushes your RAM)
# CHUNK_SIZE = 100000

# print("Loading reference frameworks...")
# with open(SPLITS_PATH, "rb") as f:
#     pc_splits = pickle.load(f)

# meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")
# protein_ids = meta_df.index
# total_proteins = len(meta_df)

# # Dynamic column matching from split columns ('split_X' -> 'X')
# target_models = [c.replace("split_", "") for c in pc_splits.columns if c.startswith("split_")]

# # Align the split reference table with our metadata index
# pc_splits = pc_splits.reindex(protein_ids)

# # Get a unique list of prediction filenames by looking at folder s_0
# folder_s0 = DATA_DIR / "random_pc_r_5_da_2_bal_s_0"
# pkl_filenames = [f.name for f in folder_s0.glob("all_preds_*.pkl")]

# print(f"Total Proteins: {total_proteins} | Target Models: {len(target_models)}")
# print(f"Processing in chunks of {CHUNK_SIZE} rows across {len(pkl_filenames)} batches.")

# # --- STEP 2: THE ROW-CHUNKED LOOP ---
# for start_idx in range(0, total_proteins, CHUNK_SIZE):
#     end_idx = min(start_idx + CHUNK_SIZE, total_proteins)
#     chunk_ids = protein_ids[start_idx:end_idx]
    
#     print(f"\n--- Processing Protein Chunk {start_idx} to {end_idx} ---")
    
#     # 1. Isolate the split mapping for just this row chunk
#     splits_chunk = pc_splits.loc[chunk_ids]
    
#     # 2. Allocate an empty DataFrame template for this chunk (uses minimal RAM)
#     combined_chunk = pd.DataFrame(
#         np.nan, 
#         index=chunk_ids, 
#         columns=target_models, 
#         dtype=np.float32
#     )
    
#     # 3. Harvest data file-by-file for this specific row chunk
#     # 3. Populate this chunk file by file
#     for file_name in tqdm(pkl_filenames, desc="Harvesting batches"):
        
#         # Loop through all 5 split folds (s_0 to s_4)
#         for k in range(5):
#             current_folder = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}"
#             file_path = current_folder / file_name
            
#             if not file_path.exists():
#                 continue
                
#             # Load the file
#             with open(file_path, "rb") as f:
#                 pred_slice = pickle.load(f)
                
#             # Drop everything except the rows in our current chunk immediately
#             pred_slice_chunk = pred_slice.loc[pred_slice.index.isin(chunk_ids)]
#             del pred_slice  # Clear from RAM
            
#             if pred_slice_chunk.empty:
#                 continue
                
#             # Process columns matching our target models
#             for col in pred_slice_chunk.columns:
#                 if col in combined_chunk.columns:
#                     split_col_name = f"split_{col}"
                    
#                     if split_col_name in splits_chunk.columns:
#                         # Create our boolean mask for rows belonging to fold k
#                         mask = (splits_chunk[split_col_name].astype(float) == k)
                        
#                         if mask.any():
#                             # Safely align indices to our master chunk template
#                             aligned_series = pred_slice_chunk[col].reindex(chunk_ids)
                            
#                             # --- FIX: Safe Extraction & NumPy Assignment ---
#                             # Extract raw values matching the mask as an explicit float32 numpy array.
#                             # This bypasses the Pandas alignment type check completely.
#                             values_to_assign = aligned_series[mask].to_numpy(dtype=np.float32)
                            
#                             # Double-check that we actually have values to assign
#                             if len(values_to_assign) > 0:
#                                 combined_chunk.loc[mask, col] = values_to_assign
#     # 4. Save this completed chunk directly to disk as a lightweight Parquet file
#     chunk_file_path = OUTPUT_DIR / f"master_chunk_{start_idx}_{end_idx}.parquet"
#     combined_chunk.to_parquet(chunk_file_path, engine="pyarrow", compression="snappy")
#     print(f"Saved chunk to {chunk_file_path}")
    
#     # Clean up variables before starting the next row chunk
#     del combined_chunk
#     del splits_chunk

# print("\nAll chunks processed successfully! No crashes.")

In [16]:
import pandas as pd
import pickle
from pathlib import Path

with open("../Data/all_random_pc_splits.pkl", "rb") as f:
    pc_splits = pickle.load(f)

sample = pd.read_pickle("../Data/random_pc_r_5_da_2_bal_s_0/all_preds_gramneg_capsid.pkl")

print("Prediction file index name:", sample.index.name)
print("Splits file index name:", pc_splits.index.name)
print("First 5 prediction file proteins:", sample.index[:5].tolist())
print("Are prediction proteins a subset of splits index?",
      sample.index.isin(pc_splits.index).all())

# Check split value distribution for these proteins, one column
split_col = "split_" + sample.columns[0]
print(f"\nSplit values for '{split_col}' among capsid proteins:")
print(pc_splits.loc[sample.index, split_col].value_counts().sort_index())

Prediction file index name: proteinID
Splits file index name: proteinID
First 5 prediction file proteins: ['AB002632_00003', 'AB002632_00005', 'AB002632_00006', 'AB002632_00007', 'AB008550_00005']
Are prediction proteins a subset of splits index? True

Split values for 'split_Vibrio_capsid' among capsid proteins:
split_Vibrio_capsid
0     8142
1     8380
2     8364
3    10346
4     9572
Name: count, dtype: Int64


---

In [17]:
# --- Build accession lookup directly from protein IDs ---
# No metadata needed: proteinID format is ACCESSION_PROTEINNUMBER
protein_to_accession = pd.Series(
    [pid.rsplit("_", 1)[0] for pid in pc_splits.index],
    index=pc_splits.index,
    name="accession"
)
print(f"Unique phages: {protein_to_accession.nunique()}")

# --- Discover all prediction columns from folder s_0 ---
# folder_s0 = DATA_DIR / "random_pc_r_5_da_2_bal_s_0"
# pkl_files = sorted(folder_s0.glob("all_preds_*.pkl"))
# --- Dynamic Prefixes Setup ---
# Scan all splits to get the unique filenames across the entire run
all_pkl_filenames = set()
for k in range(5):
    folder = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}"
    if folder.exists():
        all_pkl_filenames.update([f.name for f in folder.glob("all_preds_*.pkl")])

# Sort them to keep execution deterministic
pkl_filenames = sorted(list(all_pkl_filenames))
print(f"Discovered {len(pkl_filenames)} unique prediction files across all folds:")
print(pkl_filenames)
print(f"Prediction files per folder: {len(pkl_filenames)}")

# Verify column matching before starting
print("\nVerifying split column matching...")
# all_pred_cols = []
# for f in pkl_filenames:
#     df = pd.read_pickle(f)
#     all_pred_cols.extend(df.columns.tolist())
# all_pred_cols = list(dict.fromkeys(all_pred_cols))
# missing = [c for c in all_pred_cols if f"split_{c}" not in pc_splits.columns]
# print(f"Total prediction columns: {len(all_pred_cols)}")
# print(f"Columns missing from splits file: {len(missing)}")
# # Must be 0 before continuing

all_pred_cols = []
for filename in pkl_filenames:
    # Look for the file in folder 0 (or whichever folder has it)
    file_path = DATA_DIR / "random_pc_r_5_da_2_bal_s_0" / filename
    
    # Fallback to check other folders if split 0 doesn't have it
    if not file_path.exists():
        for k in range(1, 5):
            alternate_path = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}" / filename
            if alternate_path.exists():
                file_path = alternate_path
                break
                
    if file_path.exists():
        df = pd.read_pickle(file_path)
        all_pred_cols.extend(df.columns.tolist())
    else:
        print(f"Warning: Could not find {filename} in any split folder to extract column names.")

all_pred_cols = list(dict.fromkeys(all_pred_cols))
print(f"Collected {len(all_pred_cols)} unique prediction columns.")

Unique phages: 18474
Discovered 81 unique prediction files across all folds:
['all_preds_gramneg_DNA-associated.pkl', 'all_preds_gramneg_DNA-associated_1.pkl', 'all_preds_gramneg_DNA_polymerase.pkl', 'all_preds_gramneg_RNA-associated.pkl', 'all_preds_gramneg_adsorption-related.pkl', 'all_preds_gramneg_annealing.pkl', 'all_preds_gramneg_anti-restriction.pkl', 'all_preds_gramneg_baseplate.pkl', 'all_preds_gramneg_capsid.pkl', 'all_preds_gramneg_cell_wall_depolymerase.pkl', 'all_preds_gramneg_ejection.pkl', 'all_preds_gramneg_endolysin.pkl', 'all_preds_gramneg_head-tail_joining.pkl', 'all_preds_gramneg_helicase.pkl', 'all_preds_gramneg_holin.pkl', 'all_preds_gramneg_integration.pkl', 'all_preds_gramneg_lysin.pkl', 'all_preds_gramneg_lysis.pkl', 'all_preds_gramneg_lysis_inhibitor.pkl', 'all_preds_gramneg_nuclease.pkl', 'all_preds_gramneg_nucleotide_metabolism.pkl', 'all_preds_gramneg_packaging_assembly.pkl', 'all_preds_gramneg_phosphorylation.pkl', 'all_preds_gramneg_portal.pkl', 'all_pred

In [18]:
# # --- Main reconstruction loop ---
# # Accumulate sum and count at PHAGE level immediately
# # Never store the full protein-level matrix

# sum_df = None
# count_df = None

# for k in range(5):
#     folder = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}"
#     print(f"\n=== Split {k} ===")
    
#     for pkl_path in pkl_files:
#         file_path = folder / pkl_path.name
#         if not file_path.exists():
#             print(f"  WARNING: {pkl_path.name} missing in split {k}")
#             continue
        
#         preds = pd.read_pickle(file_path)
#         # preds: rows = proteins (subset), cols = host_function names
        
#         file_sum = {}
#         file_count = {}
        
#         for col in preds.columns:
#             split_col = f"split_{col}"
#             if split_col not in pc_splits.columns:
#                 continue
            
#             # Proteins in this file that were held out in split k
#             # Use only the intersection — prediction file is a subset of all proteins
#             shared_idx = preds.index.intersection(
#                 pc_splits.index[pc_splits[split_col] == k]
#             )
            
#             if len(shared_idx) == 0:
#                 continue
            
#             vals = preds.loc[shared_idx, col]
#             accessions = protein_to_accession.loc[shared_idx]
            
#             # Aggregate to phage level immediately
#             file_sum[col] = vals.groupby(accessions).sum()
#             file_count[col] = vals.groupby(accessions).count()
        
#         if not file_sum:
#             continue
        
#         file_sum_df = pd.DataFrame(file_sum)
#         file_count_df = pd.DataFrame(file_count)
        
#         if sum_df is None:
#             sum_df = file_sum_df
#             count_df = file_count_df
#         else:
#             sum_df = sum_df.add(file_sum_df, fill_value=0)
#             count_df = count_df.add(file_count_df, fill_value=0)
        
#         del preds, file_sum_df, file_count_df

# print("\nReconstruction complete.")
# print(f"Phage matrix shape: {sum_df.shape}")

In [ ]:
# --- Main reconstruction loop (ROBUST) ---
# Accumulate sum and count at PHAGE level immediately

sum_df = None
count_df = None

for k in range(5):
    folder = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}"
    print(f"\n=== Processing Split {k} ===")
    
    for filename in pkl_filenames:
        file_path = folder / filename
        
        # Check if this specific chunk variant exists in this fold
        if not file_path.exists():
            print(f"  Note: {filename} does not exist in Split {k} (Skipping cleanly)")
            continue
            
        try:
            preds = pd.read_pickle(file_path)
        except Exception as e:
            print(f"  Error reading {filename} in Split {k}: {e}")
            continue
            
        file_sum = {}
        file_count = {}
        
        for col in preds.columns:
            split_col = f"split_{col}"
            if split_col not in pc_splits.columns:
                continue
            
            # Find matching index intersections for fold validation
            shared_idx = preds.index.intersection(
                pc_splits.index[pc_splits[split_col] == k]
            )
            
            if len(shared_idx) == 0:
                continue
            
            vals = preds.loc[shared_idx, col]
            accessions = protein_to_accession.loc[shared_idx]
            
            # Aggregate to phage level immediately
            file_sum[col] = vals.groupby(accessions).sum()
            file_count[col] = vals.groupby(accessions).count()
        
        if not file_sum:
            continue
        
        file_sum_df = pd.DataFrame(file_sum)
        file_count_df = pd.DataFrame(file_count)
        
        if sum_df is None:
            sum_df = file_sum_df
            count_df = file_count_df
        else:
            sum_df = sum_df.add(file_sum_df, fill_value=0)
            count_df = count_df.add(file_count_df, fill_value=0)
        
        del preds, file_sum_df, file_count_df

print("\nReconstruction complete.")
print(f"Phage matrix shape: {sum_df.shape if sum_df is not None else 'Empty'}")


=== Processing Split 0 ===


In [ ]:
DATA_DIR = Path("../Data")

# Load the uncontaminated, aggregated phage-level dataset
phage_scores = pd.read_parquet(DATA_DIR / "phage_host_scores.parquet")
print("Successfully loaded engineered dataset: phage_host_scores.parquet")

# Inspect dataset dimensions and structure
print(f"\nMatrix Shape : {phage_scores.shape}")
print(f"Index Name   : {phage_scores.index.name} (Phage Accessions)")
print(f"Sample Indices: {phage_scores.index[:3].tolist()}")
print(f"Sample Columns: {phage_scores.columns[:5].tolist()}")

Successfully loaded engineered dataset: phage_host_scores.parquet

Matrix Shape : (18443, 2036)
Index Name   : accession (Phage Accessions)
Sample Indices: ['AB002632', 'AB008550', 'AB009866']
Sample Columns: ['Campylobacter_DNA-associated', 'Campylobacter_DNA_polymerase', 'Campylobacter_RNA-associated', 'Campylobacter_adsorption-related', 'Campylobacter_annealing']


In [ ]:
# Cell 2 — quality checks
nan_per_col = phage_scores.isna().mean()
print(f"NaN fraction overall:      {nan_per_col.mean():.3f}")
print(f"Completely empty columns:  {(nan_per_col == 1.0).sum()}")  # must be 0
print(f"Min NaN per col:           {nan_per_col.min():.3f}")
print(f"Max NaN per col:           {nan_per_col.max():.3f}")
print(f"Score range:               [{phage_scores.min().min():.3f}, {phage_scores.max().max():.3f}]")

# # Host names from column structure
# hosts = sorted(set(c.rsplit("_", 1)[0] for c in phage_scores.columns))
# print(f"\nUnique host genera: {len(hosts)}")
# print(hosts)

hosts = sorted(set(c.split("_", 1)[0] for c in phage_scores.columns))
print(f"\nUnique host genera: {len(hosts)}")
print(hosts)

NaN fraction overall:      0.627
Completely empty columns:  0
Min NaN per col:           0.418
Max NaN per col:           1.000
Score range:               [0.000, 1.000]

Unique host genera: 54
['Campylobacter', 'Caulobacter', 'Cellulophaga', 'Citrobacter', 'Clostridioides', 'Clostridium', 'Corynebacterium', 'Cronobacter', 'Dickeya', 'Edwardsiella', 'Enterobacter', 'Enterococcus', 'Erwinia', 'Escherichia', 'Flavobacterium', 'Gordonia', 'Haloarcula', 'Halorubrum', 'Klebsiella', 'Lacticaseibacillus', 'Lactobacillus', 'Lactococcus', 'Listeria', 'Microbacterium', 'Microcystis', 'Mycobacterium', 'Paenibacillus', 'Pantoea', 'Pectobacterium', 'Pelagibacter', 'Prochlorococcus', 'Propionibacterium', 'Proteus', 'Providencia', 'Pseudoalteromonas', 'Pseudomonas', 'Ralstonia', 'Rhizobium', 'Rhodobacter', 'Roseobacter', 'Salmonella', 'Serratia', 'Shewanella', 'Shigella', 'Sinorhizobium', 'Staphylococcus', 'Stenotrophomonas', 'Streptococcus', 'Streptomyces', 'Sulfolobus', 'Synechococcus', 'Vibrio', '

In [ ]:
print("Checking for missing prediction files across all 5 folders...\n")
any_missing = False
for k in range(5):
    folder = DATA_DIR / f"random_pc_r_5_da_2_bal_s_{k}"
    missing = [f.name for f in pkl_files if not (folder / f.name).exists()]
    if missing:
        any_missing = True
        print(f"Split {k}: {len(missing)} missing file(s): {missing}")
    else:
        print(f"Split {k}: complete ({len(pkl_files)} files)")

if not any_missing:
    print("\nNo other missing files — DNA-associated_1 is the only known gap.")

Checking for missing prediction files across all 5 folders...

Split 0: complete (81 files)
Split 1: 1 missing file(s): ['all_preds_gramneg_DNA-associated_1.pkl']
Split 2: 1 missing file(s): ['all_preds_gramneg_DNA-associated_1.pkl']
Split 3: 1 missing file(s): ['all_preds_gramneg_DNA-associated_1.pkl']
Split 4: 1 missing file(s): ['all_preds_gramneg_DNA-associated_1.pkl']


In [ ]:
actual_hosts = sorted(set(c.split("_")[0] for c in phage_scores.columns))
print(f"Actual host genera: {len(actual_hosts)}")
print(actual_hosts)

Actual host genera: 54
['Campylobacter', 'Caulobacter', 'Cellulophaga', 'Citrobacter', 'Clostridioides', 'Clostridium', 'Corynebacterium', 'Cronobacter', 'Dickeya', 'Edwardsiella', 'Enterobacter', 'Enterococcus', 'Erwinia', 'Escherichia', 'Flavobacterium', 'Gordonia', 'Haloarcula', 'Halorubrum', 'Klebsiella', 'Lacticaseibacillus', 'Lactobacillus', 'Lactococcus', 'Listeria', 'Microbacterium', 'Microcystis', 'Mycobacterium', 'Paenibacillus', 'Pantoea', 'Pectobacterium', 'Pelagibacter', 'Prochlorococcus', 'Propionibacterium', 'Proteus', 'Providencia', 'Pseudoalteromonas', 'Pseudomonas', 'Ralstonia', 'Rhizobium', 'Rhodobacter', 'Roseobacter', 'Salmonella', 'Serratia', 'Shewanella', 'Shigella', 'Sinorhizobium', 'Staphylococcus', 'Stenotrophomonas', 'Streptococcus', 'Streptomyces', 'Sulfolobus', 'Synechococcus', 'Vibrio', 'Xanthomonas', 'Yersinia']
